In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Mon Jul 12 10:02:03 2021

@author: hudew
"""

# I did not design or create the DDPM model, I only implemented it in this script

import os
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.transforms import ToTensor, Resize, Compose
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim, mean_squared_error as mse
import cv2
from tqdm import tqdm

from DDPM_GaussianDiffusion import GaussianDiffusion, get_beta_schedule
from DDPM_Net import Model

MODEL_DIR = '../ckpts/'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BETA_START = 0.00001
BETA_END = 0.00006
NUM_DIFFUSION_TIMESTEPS = 100
ALIGNMENT_SCORE_THRESHOLD = 0.3  # SSIM threshold for alignment score

def load_images_from_folder(folder, size=(512, 512)):
    images = []
    filenames = []
    transform = Compose([
        Resize(size),  
        ToTensor()   
    ])
    
    for filename in sorted(os.listdir(folder)):  # Sort to maintain consistent order
        if filename.lower().endswith(".tiff"):
            img_path = os.path.join(folder, filename)
            with Image.open(img_path) as img:
                img_tensor = transform(img.convert('L'))  # Convert to grayscale and apply transformations
                images.append(img_tensor)
                filenames.append(filename)
    return images, filenames

# Function to initialize and load the model
def load_model():
    betas = get_beta_schedule('linear', beta_start=BETA_START, beta_end=BETA_END, num_diffusion_timesteps=NUM_DIFFUSION_TIMESTEPS)
    betas = torch.from_numpy(betas).float().to(DEVICE)

    model = Model().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_DIR + "DDPM_oct_dataset2_2021-07-08.pt"))
    model.eval()
    
    return model, betas

def fuse_images(images, filenames, num_neighbors=5):
    fused_images = []

    num_images = len(images)
    for i in range(2, num_images):  # Skip the first two images
        # Apply self-fusion
        fused_image = torch.zeros_like(images[i])
        count = torch.zeros_like(images[i])
        
        # Define the range of neighbors to consider
        start = max(0, i - num_neighbors)
        end = min(num_images, i + num_neighbors + 1)

        for j in range(start, end):
            if i != j and j >= 2:  # Skip the first two images
                fused_image += images[j]
                count += 1

        fused_image = (fused_image / count).clamp(min=0, max=1)
        fused_images.append(fused_image)

    return fused_images

def fuse_and_denoise_images_neighbours_ddpm(images, filenames, model, betas, num_neighbors=20):
    diffusion = GaussianDiffusion(betas=betas, device=DEVICE)
    fused_denoised_images = []

    num_images = len(images)
    for i in range(2, num_images):  # Skip the first two images
        # Apply self-fusion
        fused_image = torch.zeros_like(images[i])
        count = torch.zeros_like(images[i])
        
        # Define the range of neighbors to consider
        start = max(0, i - num_neighbors)
        end = min(num_images, i + num_neighbors + 1)

        for j in range(start, end):
            if i != j and j >= 2:  # Skip the first two images
                fused_image += images[j]
                count += 1

        fused_image = (fused_image / count).clamp(min=0, max=1)

        # Denoise the self-fused image
        fused_image = fused_image.unsqueeze(0).to(DEVICE)  # Add batch dimension and send to device
        t = torch.tensor([NUM_DIFFUSION_TIMESTEPS // 2], device=DEVICE)  # Example timestep

        with torch.no_grad():
            eps_t = model(fused_image, t)
            img_denoised = diffusion.denoise(fused_image, eps_t, t)
            fused_denoised_images.append(img_denoised.squeeze(0).cpu())  # Remove batch dimension and transfer to CPU

    return fused_denoised_images

# Define function to apply self-fusion and denoise images using the model
def fuse_and_denoise_images_all_ddpm(images, filenames, model, betas):
    diffusion = GaussianDiffusion(betas=betas, device=DEVICE)
    fused_denoised_images = []

    for i in range(2, len(images)):  # Skip the first two images
        fused_image = torch.zeros_like(images[i])
        count = torch.zeros_like(images[i])
        
        for j in range(2, len(images)):  # Skip the first two images
            if i != j:
                fused_image += images[j]
                count += 1

        fused_image = (fused_image / count).clamp(min=0, max=1)

        fused_image = fused_image.unsqueeze(0).to(DEVICE)
        t = torch.tensor([NUM_DIFFUSION_TIMESTEPS // 2], device=DEVICE) 

        with torch.no_grad():
            eps_t = model(fused_image, t)
            img_denoised = diffusion.denoise(fused_image, eps_t, t)
            fused_denoised_images.append(img_denoised.squeeze(0).cpu()) 

    return fused_denoised_images

def calculate_psnr(original, denoised):
    """Calculate Peak Signal-to-Noise Ratio between two images."""
    return psnr(original, denoised, data_range=denoised.max() - denoised.min())

def calculate_ssim(original, denoised):
    """Calculate Structural Similarity Index between two images."""
    return ssim(original, denoised, data_range=denoised.max() - denoised.min())

def calculate_mse(original, denoised):
    """Calculate Mean Squared Error between two images."""
    return mse(original, denoised)

def evaluate_denoising(original_images, denoised_images):
    psnr_values = []
    ssim_values = []
    mse_values = []

    for original, denoised in zip(original_images, denoised_images):
        if isinstance(original, torch.Tensor):
            original = original.detach().cpu().numpy()
        if isinstance(denoised, torch.Tensor):
            denoised = denoised.detach().cpu().numpy()

        original = original.squeeze() 
        denoised = denoised.squeeze()

        psnr_values.append(calculate_psnr(original, denoised))
        ssim_values.append(calculate_ssim(original, denoised))
        mse_values.append(calculate_mse(original, denoised))

    # Print average values for all images
    print("Average PSNR:", np.mean(psnr_values))
    print("Average SSIM:", np.mean(ssim_values))
    print("Average MSE:", np.mean(mse_values))

    return psnr_values, ssim_values, mse_values

def plot_images(original_images, denoised_images):

    plt.figure(figsize=(30, 20)) # Plot side by side
    num_images = min(len(original_images), 5)  # Limit the number of images displayed
    
    for i in range(num_images):
        plt.subplot(2, num_images, i + 1)
        plt.imshow(original_images[i].squeeze().numpy(), cmap='gray')
        plt.title('Original Image')
        plt.axis('off')

        plt.subplot(2, num_images, num_images + i + 1)
        plt.imshow(denoised_images[i].squeeze().numpy(), cmap='gray')
        plt.title('Denoised Image')
        plt.axis('off')

    plt.show()

def save_denoised_images(denoised_images, filenames, output_dir):
    for img, filename in zip(denoised_images, filenames[2:]):  # Skip the first two filenames
        img = img.squeeze().numpy()  # Remove channel dimension if it's 1
        img = (img * 255).astype(np.uint8)  # Convert to uint8 format
        img_pil = Image.fromarray(img)
        output_path = os.path.join(output_dir, filename)
        img_pil.save(output_path)

def save_original_images(images, filenames, output_dir):
    for img, filename in zip(images[2:], filenames[2:]):  # Skip the first two images and filenames
        img = img.squeeze().numpy()  # Remove channel dimension if it's 1
        img = (img * 255).astype(np.uint8)  # Convert to uint8 format
        img_pil = Image.fromarray(img)
        output_path = os.path.join(output_dir, filename)
        img_pil.save(output_path)

def full_fusion_ddpm(images, filenames):
    model, betas = load_model()
    fused_denoised_images = fuse_and_denoise_images_all_ddpm(images, filenames, model, betas)
    evaluate_denoising(images[2:], fused_denoised_images)  # Skip the first two images
    plot_images(images[2:], fused_denoised_images)  # Skip the first two images
    return fused_denoised_images, f"../outputs/{patient}/full_fusion_DDPM"
    
def full_fusion_with_neighbours_ddpm(images, filenames):
    model, betas = load_model()
    fused_denoised_images = fuse_and_denoise_images_neighbours_ddpm(images, filenames, model, betas)
    evaluate_denoising(images[2:], fused_denoised_images)  # Skip the first two images
    plot_images(images[2:], fused_denoised_images)  # Skip the first two images
    return fused_denoised_images, f"../outputs/{patient}/neighbour_fusion_DDPM"

def segment_bright_parts(image):
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    _, segmented_image = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return segmented_image

def calculate_overlap(segmented_base_image, segmented_provided_image):
    intersection = np.logical_and(segmented_base_image, segmented_provided_image)
    return np.sum(intersection)

def optimize_translation(segmented_base_image, segmented_provided_image, max_shift=30):
    best_overlap = 0
    best_translation = (0, 0)
    for dx in range(-max_shift, max_shift+1):
        for dy in range(-max_shift, max_shift+1):
            M = np.float32([[1, 0, dx], [0, 1, dy]])
            translated_image = cv2.warpAffine(segmented_provided_image, M, (segmented_provided_image.shape[1], segmented_provided_image.shape[0]))
            overlap = calculate_overlap(segmented_base_image, translated_image)
            if overlap > best_overlap:
                best_overlap = overlap
                best_translation = (dx, dy)
    return best_translation

def apply_translation(image, translation):
    dx, dy = translation
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    translated_image = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))
    return translated_image

def crop_to_bounding_box(image, bounding_box):
    x0, y0, x1, y1 = bounding_box
    cropped_image = image[x0:x1+1, y0:y1+1]
    return cropped_image

def get_non_black_bounding_box(image):
    mask = image > 0
    coords = np.column_stack(np.where(mask))
    x0, y0 = coords.min(axis=0)
    x1, y1 = coords.max(axis=0)
    return x0, y0, x1, y1

def get_intersection_bounding_box(box1, box2):
    x0 = max(box1[0], box2[0])
    y0 = max(box1[1], box2[1])
    x1 = min(box1[2], box2[2])
    y1 = min(box1[3], box2[3])
    return x0, y0, x1, y1

def calculate_alignment_score(image1, image2):
    """Calculate alignment score using SSIM."""
    score, _ = ssim(image1, image2, full=True)
    return score

def process_and_align_images(base_folder, provided_folder, patient):
    output_base_folder = f"C:\\Users\\CL-11\\OneDrive\\Research\\OCT_Image_Rego_Denoising\\outputs\\{patient}\\cropped_base_images"
    output_provided_folder = f"C:\\Users\\CL-11\\OneDrive\\Research\\OCT_Image_Rego_Denoising\\outputs\\{patient}\\aligned_self_fused_images"
    
    os.makedirs(output_base_folder, exist_ok=True)
    os.makedirs(output_provided_folder, exist_ok=True)

    base_images = sorted(os.listdir(base_folder))
    provided_images = sorted(os.listdir(provided_folder))

    for base_image_name, provided_image_name in tqdm(zip(base_images, provided_images)): # tqdm
        base_image_path = os.path.join(base_folder, base_image_name)
        provided_image_path = os.path.join(provided_folder, provided_image_name)

        base_image = cv2.imread(base_image_path, cv2.IMREAD_GRAYSCALE)
        provided_image = cv2.imread(provided_image_path, cv2.IMREAD_GRAYSCALE)
        provided_image = cv2.resize(provided_image, (base_image.shape[1], base_image.shape[0]))

        segmented_base_image = segment_bright_parts(base_image)
        segmented_provided_image = segment_bright_parts(provided_image)

        best_translation = optimize_translation(segmented_base_image, segmented_provided_image)
        aligned_image = apply_translation(provided_image, best_translation)

        bounding_box_base = get_non_black_bounding_box(base_image)
        bounding_box_aligned = get_non_black_bounding_box(aligned_image)
        intersection_bounding_box = get_intersection_bounding_box(bounding_box_base, bounding_box_aligned)

        cropped_base_image = crop_to_bounding_box(base_image, intersection_bounding_box)
        cropped_aligned_image = crop_to_bounding_box(aligned_image, intersection_bounding_box)

        alignment_score = calculate_alignment_score(cropped_base_image, cropped_aligned_image)
        
        if alignment_score < ALIGNMENT_SCORE_THRESHOLD:
            print(f"Skipping poorly aligned pair: {base_image_name}, {provided_image_name} with alignment score: {alignment_score}")
            continue
        
        output_base_path = os.path.join(output_base_folder, base_image_name)
        output_provided_path = os.path.join(output_provided_folder, provided_image_name)
        cv2.imwrite(output_base_path, cropped_base_image)
        cv2.imwrite(output_provided_path, cropped_aligned_image)

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.title(f'Cropped Base Image: {base_image_name}')
        plt.imshow(cropped_base_image, cmap='gray')
        plt.subplot(1, 2, 2)
        plt.title(f'Cropped Aligned Image: {provided_image_name}')
        plt.imshow(cropped_aligned_image, cmap='gray')
        plt.show()

if __name__ == "__main__":
    for i in range(4, 30):
        patient = f'RawDataQA ({i})'  # Change this to the patient ID, this is basically an argument but I prefer notebooks



        if not os.path.exists(f"../outputs/{patient}"):
            os.mkdir(f"../outputs/{patient}")
            os.mkdir(f"../outputs/{patient}/neighbour_fusion_DDPM")
            os.mkdir(f"../outputs/{patient}/original")

        folder_path = f"C:\\Datasets\\ICIP training data\\ICIP training data\\0\\{patient}"
        output_path = f"C:\\Users\\CL-11\\OneDrive\\Research\\OCT_Denoising\\data\\denoised\\{patient}"

        original_images, filenames = load_images_from_folder(folder_path)

        # Save original images
        save_original_images(original_images, filenames, f"../outputs/{patient}/original")

        #fused_denoised_images, output_path = full_fusion_ddpm(original_images, filenames)
        fused_denoised_images, output_path = full_fusion_with_neighbours_ddpm(original_images, filenames)

        save_denoised_images(fused_denoised_images, filenames, output_path)

        base_folder = f'C:\\Users\\CL-11\\OneDrive\\Research\\OCT_Image_Rego_Denoising\\outputs\\{patient}\\original'
        provided_folder = f"C:\\Users\\CL-11\\OneDrive\\Research\\OCT_Image_Rego_Denoising\\outputs\\{patient}\\neighbour_fusion_DDPM"

        process_and_align_images(base_folder, provided_folder, patient)
